# YouTube QA Bot (LangChain + Gemini + FAISS) 

This project lets you ask questions about a YouTube video by:
1. Downloading the video transcript
2. Splitting it into chunks
3. Embedding the chunks using Google Vertex AI embeddings
4. Storing them in a FAISS vector database
5. Querying the most relevant chunks
6. Generating answers using Gemini (Google Generative AI)

---

## 🚀 Features

- YouTube transcript loading via LangChain
- Text chunking with overlap for better context
- Vector search using FAISS
- Semantic search over video content
- Question answering using Gemini (`gemini-2.5-flash`)
- Strict "context-only" answering (no hallucinations)

---

## 📦 Tech Stack

- LangChain
- Google Gemini API (`ChatGoogleGenerativeAI`)
- Google Vertex AI Embeddings
- FAISS (vector database)
- YouTube Transcript Loader

---

## 1. Import libraries

In [1]:
from dotenv import load_dotenv
import os
import textwrap

from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_vertexai import VertexAIEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

C:\Users\luigi\AppData\Local\Temp\ipykernel_5184\1973024199.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import YoutubeLoader


## 2. Load environment variables

In [2]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")

## 3. Initialize embeddings (Vertex AI)

In [3]:
# Make sure you ran:
# gcloud auth application-default login

embeddings = VertexAIEmbeddings(
    model_name="text-embedding-005",
    project="68582664605",   # <-- replace with your project id
    location="us-central1"
)

C:\Users\luigi\AppData\Local\Temp\ipykernel_5184\4075384302.py:4: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embeddings = VertexAIEmbeddings(
C:\Users\luigi\AppData\Local\Temp\ipykernel_5184\4075384302.py:4: LangChainDeprecationWarning: The class `VertexAIEmbeddings` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import GoogleGenerativeAIEmbeddings``.
  embeddings = VertexAIEmbeddings(


## 4. Function: Load YouTube transcript + build FAISS DB

In [4]:
def create_db_from_youtube_video_url(video_url: str):

    loader = YoutubeLoader.from_youtube_url(
        video_url,
        add_video_info=False
    )

    transcript = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=100
    )

    docs = text_splitter.split_documents(transcript)

    db = FAISS.from_documents(docs, embeddings)

    return db

## 5. Function: Ask questions from the video

In [5]:
def get_response_from_query(db, query: str, k: int = 4):

    docs = db.similarity_search(query, k=k)

    docs_page_content = "\n\n".join([doc.page_content for doc in docs])

    model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.2
    )

    prompt = ChatPromptTemplate.from_template(
        """
You are a helpful assistant that answers questions about a YouTube video
using only the provided transcript context.

Context:
{docs}

Question:
{question}

Rules:
- Only answer from the transcript
- If the transcript does not contain the answer, say "I don't know"
"""
    )

    chain = prompt | model | StrOutputParser()

    response = chain.invoke({
        "docs": docs_page_content,
        "question": query
    })

    return response, docs

## 6. Load a YouTube video

In [6]:
video_url = "https://www.youtube.com/watch?v=2P27Ef-LLuQ"

db = create_db_from_youtube_video_url(video_url)

print("Vector DB created successfully!")

Vector DB created successfully!


## 7. Ask a question

In [ ]:
query = "what are they saying about Impact on Jobs?"

response, docs = get_response_from_query(db, query)

print(textwrap.fill(response, width=80))

Regarding the impact on jobs, the transcript mentions a technical copywriter
whose job evolved from managing a team of representatives to managing bots.
Eventually, once the bots were sufficiently trained, the copywriter was "out."
The speaker agrees that it's clear people will be managing "a lot of AIs" doing
different tasks. While the speaker is "not a job doomer," they anticipate the
transition will likely be "rough" in some cases. They believe humans are deeply
wired to care about other people, relative status, being of use and service, and
expressing creative spirit, suggesting these fundamental drivers won't
disappear. They expect that what people do all day in 2050 will look very
different from today, but they don't foresee a future without meaning or a
broken economy. The speaker even expresses enthusiasm for the idea of an "AI CEO
of Open AI."  The transcript does not explicitly use the term "agents" in a way
that distinguishes them from "bots" or "AIs," but it discusses peopl